### Load env

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [2]:
import pandas as pd

# Set to None to show the full content of each cell without truncation
pd.set_option('display.max_colwidth', None)

# To ensure the full DataFrame width is used before wrapping to a new 'page'
pd.set_option('display.width', 1000)


### roshansk23/Vietnam_HighSchool_Exam_Dataset Vai trò: nguồn exams_mcq chính cho đề thi THPT kiểu MCQ.
### Dataset này có khoảng 6,663 mẫu, 8 môn, với các field quan trọng như question, options, answer

In [3]:

df_Vietnam_HighSchool_Exam_Dataset = pd.read_json("hf://datasets/roshansk23/Vietnam_HighSchool_Exam_Dataset/cleaned_VNHSGE.json")

In [4]:
df_Vietnam_HighSchool_Exam_Dataset.head(1)

,language,country,file_name,source,license,level,category_en,category_original_lang,original_question_num,question,options,answer
0,vi,vietnam,His_3.json,https://github.com/Xdao85/VNHSGE,open,high school,History,Lịch sử,Unknown,Đặc điểm nổi bật của nền kinh tế Mĩ sau chiến tranh thế giới thứ 2 là,"[bị thiệt hại nặng nề về người và của do hậu quả của chiến tranh thế giới thứ hại., phát triển mạnh mẽ, vươn lên hàng thứ 2 thế giới sau Liên Xô., phát triển mạnh mẽ trở thành trung tâm kinh tế - tài chính lớn nhất thế giới., bị suy giảm nghiêm trọng vì phải lo chi phí cho sản xuất vụ khí.]",3


### hllj/vi_grade_school_math_mcq
### Vai trò: bổ sung thêm MCQ Toán tiếng Việt. Dataset này có khoảng 2,733 mẫu, với schema rất tiện cho bạn: question, choices, answer, explanation. Card cũng nói rõ dữ liệu chưa clean hoàn toàn, nên vẫn cần QC sau khi convert.

In [5]:

df_vi_grade_school_math_mcq = pd.read_json("hf://datasets/hllj/vi_grade_school_math_mcq/vietjack.json")

In [6]:
df_vi_grade_school_math_mcq.head(1)

,id,grade,url,title,problems
0,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,"[{'question': 'Câu 1: Độ dài của chiếc bút xoá là:', 'choices': ['A. 10 cm', 'B. 8 cm', 'C. 9 cm', 'D. 7 cm'], 'explanation': 'Hướng dẫn giải Đáp án đúng là: C Quan sát hình vẽ ta thấy độ dài của chiếc bút xoá là 9 cm .'}, {'question': 'Câu 2: Nam cao 98 cm, Minh cao 92 cm, Mai cao 88 cm và An cao 89 cm. Bạn thấp nhất là:', 'choices': ['A. Nam', 'B. Minh', 'C. Mai', 'D. An'], 'explanation': 'Hướng dẫn giải Đáp án đúng là: C So sánh chiều cao của 4 bạn, ta thấy: 88 < 89 < 92 < 98 . Trong 4 số trên số nhỏ nhất là 88, tương ứng với chiều cao của bạn Mai . Vậy bạn thấp nhất là bạn Mai .'}, {'question': 'Câu 3: Chiều cao của bạn thỏ là:', 'choices': ['A. 10 cm', 'B. 12 cm', 'C. 14 cm', 'D. 15 cm'], 'explanation': 'Hướng dẫn giải Đáp án đúng là: D Chiều cao của bạn thỏ bằng chiều cao của 3 củ cà rốt . Chiều cao của 1 củ cà rốt là 5 cm . C hiều cao của bạn thỏ là: 5 + 5 + 5 = 15 (cm) Đáp số: 15 cm.'}, {'question': 'Câu 4: An đi từ nhà đến trường hết 1 giờ. An bắt đầu đi từ nhà đến trường lúc 6 giờ sáng và đến trường lúc:', 'choices': ['A. 7 giờ sáng', 'B. 7 giờ tối', 'C. 8 giờ sáng', 'D. 8 giờ tối'], 'explanation': 'Hướng dẫn giải Đáp án đúng là: A Ta có: 6 giờ sáng + 1 giờ = 7 giờ sáng . Vậy An đến trường lúc 7 giờ sáng .'}, {'question': 'Câu 5: Giờ vào học buổi sáng là 7 giờ. An đến sớm hơn 1 giờ. An đến lớp lúc:', 'choices': ['A. 6 giờ', 'B. 7 giờ', 'C. 8 giờ', 'D. 9 giờ'], 'explanation': 'Hướng dẫn giải Đáp án đúng là: A Ta có: 7 giờ − 1 giờ = 6 giờ Vậy An đến lớp lúc 6 giờ .'}]"


- df_vi_grade_school_math_mcq không có cột answer như df phía trên.

- answer sẽ là: "Hướng dẫn giải Đáp án đúng là: (choices) + (explain)"

### Chuẩn hoá

In [7]:
# kiểm tra số lượng đáp án của df_Vietnam_HighSchool_Exam_Dataset
df_Vietnam_HighSchool_Exam_Dataset["answer"].value_counts(dropna=False).sort_index()

answer
1    1785
2    1693
3    1694
4    1491
Name: count, dtype: int64

- vậy 100% là đáp án A B C D

In [8]:
import ast
import re

df_ro = df_Vietnam_HighSchool_Exam_Dataset.copy()

In [87]:
# df_ro.head(1)

In [9]:
df_ro.info()

<class 'pandas.DataFrame'>
RangeIndex: 6663 entries, 0 to 6662
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   language                6663 non-null   str   
 1   country                 6663 non-null   str   
 2   file_name               6663 non-null   str   
 3   source                  6663 non-null   str   
 4   license                 6663 non-null   str   
 5   level                   6663 non-null   str   
 6   category_en             6663 non-null   str   
 7   category_original_lang  6663 non-null   str   
 8   original_question_num   6663 non-null   str   
 9   question                6663 non-null   str   
 10  options                 6663 non-null   object
 11  answer                  6663 non-null   int64 
dtypes: int64(1), object(1), str(10)
memory usage: 2.3+ MB


In [10]:
def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
    return x

df_ro["options"] = df_ro["options"].apply(ensure_list)


In [11]:
df_ro.head(1)

,language,country,file_name,source,license,level,category_en,category_original_lang,original_question_num,question,options,answer
0,vi,vietnam,His_3.json,https://github.com/Xdao85/VNHSGE,open,high school,History,Lịch sử,Unknown,Đặc điểm nổi bật của nền kinh tế Mĩ sau chiến tranh thế giới thứ 2 là,"[bị thiệt hại nặng nề về người và của do hậu quả của chiến tranh thế giới thứ hại., phát triển mạnh mẽ, vươn lên hàng thứ 2 thế giới sau Liên Xô., phát triển mạnh mẽ trở thành trung tâm kinh tế - tài chính lớn nhất thế giới., bị suy giảm nghiêm trọng vì phải lo chi phí cho sản xuất vụ khí.]",3


In [12]:
# chỉ giữ mẫu có đúng 4 lựa chọn
df_ro = df_ro[df_ro["options"].apply(lambda x: isinstance(x, list) and len(x) == 4)].copy()

In [13]:
# map answer 1-4 -> A-D
answer_map_1_based = {1: "A", 2: "B", 3: "C", 4: "D", "1": "A", "2": "B", "3": "C", "4": "D"}
df_ro["answer_letter"] = df_ro["answer"].map(answer_map_1_based)

# tách 4 lựa chọn
df_ro["choice_A"] = df_ro["options"].apply(lambda x: str(x[0]).strip())
df_ro["choice_B"] = df_ro["options"].apply(lambda x: str(x[1]).strip())
df_ro["choice_C"] = df_ro["options"].apply(lambda x: str(x[2]).strip())
df_ro["choice_D"] = df_ro["options"].apply(lambda x: str(x[3]).strip())

df_ro["subject"] = df_ro["category_original_lang"].fillna(df_ro["category_en"])
df_ro["grade"] = df_ro["level"]
df_ro["source_dataset"] = "roshansk23/Vietnam_HighSchool_Exam_Dataset"
df_ro["sample_id"] = "roshansk_" + df_ro.index.astype(str)

df_ro[["question", "choice_A", "choice_B", "choice_C", "choice_D", "answer", "answer_letter", "subject"]].head()

,question,choice_A,choice_B,choice_C,choice_D,answer,answer_letter,subject
0,Đặc điểm nổi bật của nền kinh tế Mĩ sau chiến tranh thế giới thứ 2 là,bị thiệt hại nặng nề về người và của do hậu quả của chiến tranh thế giới thứ hại.,"phát triển mạnh mẽ, vươn lên hàng thứ 2 thế giới sau Liên Xô.",phát triển mạnh mẽ trở thành trung tâm kinh tế - tài chính lớn nhất thế giới.,bị suy giảm nghiêm trọng vì phải lo chi phí cho sản xuất vụ khí.,3,C,Lịch sử
1,Nhân tố quan trọng hàng đầu giúp các nước Tây Âu nhanh chóng khôi phục kinh tế sau chiến tranh thế giới thứ hai là gỉ?,Thực hiện các cải cách dân chủ tiến bộ.,Xâm lược trở lại các thuộc địa của mình.,Nhận viện trợ của Mỹ thông qua kế hoạch Mác-san.,Củng cố chính quyền của giai cấp tư sản.,3,C,Lịch sử
2,Đâu là nguyên nhân chung cơ bản dẫn đến 3 trung tâm kinh tế tài chính Mĩ – Tây Âu – Nhật Bản khủng hoảng suy thoái kéo dài trong giai đoạn 1973 - 1991?,Tác động của khủng hoảng năng lượng năm 1973.,Sự cạnh tranh quyết liệt của các nước công nghiệp mới.,Sự chi phối ảnh hưởng của trật thế giới 2 cực và chiến tranh lạnh.,Kinh tế Mĩ suy thoái kéo theo kinh tế Nhật Bản và Tây Âu,1,A,Lịch sử
3,Việc tìm cách trở lại các thuộc địa cũ sau chiến tranh thế giới thứ 2 của các nước Tây Âu đã ảnh hưởng như thế nào đến Việt Nam?,Thực dân Pháp quay trở lại xâm lược nước ta lần thứ hai buộc nhân dân ta phải đứng lên kháng chiến chống Pháp.,Chính phủ Pháp công nhận Việt Nam là một quốc gia tự do nằm trong khối Liên hiệp Pháp.,"Ngay từ 1945, Pháp – Mỹ đã liên kết lại với nhau để chống cách mạng Việt Nam.","Không ảnh hưởng gì đến Việt Nam vì ngày 2/9/1945, nước Việt Nam Dân chủ Cộng hòa đã tuyên bố thành lập.",1,A,Lịch sử
4,Những thắng lợi nào sau đây đánh dấu chủ nghĩa thực dân cũ ở châu Phi cùng hệ thống thuộc địa của nó cơ bản bị tan rã?,"Thắng lợi của nhân dân Môdămbích, Nam Phi.","Thắng lợi của nhân dân Ai Cập, Angiêri.",Thắng lợi của nhân dân Môdămbích và Ănggôla.,"Thắng lợi của nhân dân Ai Cập, Môdămbích.",3,C,Lịch sử


###

In [14]:
df_hllj_flat = df_vi_grade_school_math_mcq.copy()
df_hllj_flat = df_hllj_flat.explode("problems", ignore_index=True)

problem_cols = pd.json_normalize(df_hllj_flat["problems"])
df_hllj_flat = pd.concat(
    [df_hllj_flat.drop(columns=["problems"]).reset_index(drop=True), problem_cols.reset_index(drop=True)],
    axis=1
)

df_hllj_flat.head(3)

,id,grade,url,title,question,choices,explanation
0,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,Câu 1: \n \n Độ dài của chiếc bút xoá là:,"[A. 10 cm, B. 8 cm, C. 9 cm, D. 7 cm]",Hướng dẫn giải \n Đáp án đúng là: C \n Quan sát hình vẽ ta thấy độ dài của chiếc bút xoá là 9 cm .
1,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,"Câu 2: \n \n Nam cao 98 cm, Minh cao 92 cm, Mai cao 88 cm và An cao 89 cm. Bạn thấp nhất là:","[A. Nam, B. Minh, C. Mai, D. An]","Hướng dẫn giải \n Đáp án đúng là: C \n So sánh chiều cao của 4 bạn, ta thấy: 88 < 89 < 92 < 98 . \n Trong 4 số trên số nhỏ nhất là 88, tương ứng với chiều cao của bạn Mai . \n Vậy bạn thấp nhất là bạn Mai ."
2,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,Câu 3: \n \n Chiều cao của bạn thỏ là:,"[A. 10 cm, B. 12 cm, C. 14 cm, D. 15 cm]",Hướng dẫn giải \n Đáp án đúng là: D \n Chiều cao của bạn thỏ bằng chiều cao của 3 củ cà rốt . \n Chiều cao của 1 củ cà rốt là 5 cm . \n C hiều cao của bạn thỏ là: \n 5 + 5 + 5 = 15 (cm) \n Đáp số: 15 cm.


In [15]:
df_hllj = df_hllj_flat.copy()

In [16]:
# parse choices thành A/B/C/D và bỏ prefix "A. ", "B. "...
# def parse_choices(choice_list):
#     if not isinstance(choice_list, list) or len(choice_list) != 4:
#         return pd.Series([None, None, None, None])
    
#     cleaned = []
#     for c in choice_list:
#         c = str(c).strip()
#         c = re.sub(r"^\s*[ABCD][\.\)]\s*", "", c)
#         cleaned.append(c)
#     return pd.Series(cleaned)

def clean_choice_text(x):
    if pd.isna(x):
        return None

    text = str(x).strip()

    # Normalize khoảng trắng đặc biệt
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    # Xóa prefix dạng:
    # A. 605
    # A . 605
    # A) 605
    # A ) 605
    # A: 605
    # A - 605
    text = re.sub(
        r"^\s*[A-Da-d]\s*[\.\)\:\-]\s*",
        "",
        text
    ).strip()

    # Normalize lại sau khi xóa prefix
    text = re.sub(r"\s+", " ", text).strip()

    return text


def parse_choices(choice_list):
    if not isinstance(choice_list, list) or len(choice_list) != 4:
        return pd.Series([None, None, None, None])

    cleaned = [clean_choice_text(c) for c in choice_list]

    return pd.Series(cleaned)

# df_hllj[["choice_A", "choice_B", "choice_C", "choice_D"]] = df_hllj["choices"].apply(parse_choices)

In [17]:
df_hllj[["choice_A", "choice_B", "choice_C", "choice_D"]] = df_hllj["choices"].apply(parse_choices)

In [18]:
def has_choice_prefix(x):
    if pd.isna(x):
        return False
    return bool(re.match(r"^\s*[ABCD]\s*[\.\)\:\-]\s*", str(x), flags=re.IGNORECASE))

for col in ["choice_A", "choice_B", "choice_C", "choice_D"]:
    n_bad = df_hllj[col].apply(has_choice_prefix).sum()
    print(f"{col} still has prefix: {n_bad}")

choice_A still has prefix: 3
choice_B still has prefix: 7
choice_C still has prefix: 1
choice_D still has prefix: 6


In [19]:
choice_cols = ["choice_A", "choice_B", "choice_C", "choice_D"]

for col in choice_cols:
    df_hllj[col] = df_hllj[col].apply(clean_choice_text)

In [20]:
prefix_pattern = r"^\s*[A-Da-d]\s*[\.\)\:\-]\s*"

for col in choice_cols:
    n_bad = df_hllj[col].astype(str).str.contains(prefix_pattern, regex=True, na=False).sum()
    print(col, "bad prefix:", n_bad)

choice_A bad prefix: 0
choice_B bad prefix: 0
choice_C bad prefix: 0
choice_D bad prefix: 0


In [21]:
df_hllj.head(3)

,id,grade,url,title,question,choices,explanation,choice_A,choice_B,choice_C,choice_D
0,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,Câu 1: \n \n Độ dài của chiếc bút xoá là:,"[A. 10 cm, B. 8 cm, C. 9 cm, D. 7 cm]",Hướng dẫn giải \n Đáp án đúng là: C \n Quan sát hình vẽ ta thấy độ dài của chiếc bút xoá là 9 cm .,10 cm,8 cm,9 cm,7 cm
1,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,"Câu 2: \n \n Nam cao 98 cm, Minh cao 92 cm, Mai cao 88 cm và An cao 89 cm. Bạn thấp nhất là:","[A. Nam, B. Minh, C. Mai, D. An]","Hướng dẫn giải \n Đáp án đúng là: C \n So sánh chiều cao của 4 bạn, ta thấy: 88 < 89 < 92 < 98 . \n Trong 4 số trên số nhỏ nhất là 88, tương ứng với chiều cao của bạn Mai . \n Vậy bạn thấp nhất là bạn Mai .",Nam,Minh,Mai,An
2,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,Câu 3: \n \n Chiều cao của bạn thỏ là:,"[A. 10 cm, B. 12 cm, C. 14 cm, D. 15 cm]",Hướng dẫn giải \n Đáp án đúng là: D \n Chiều cao của bạn thỏ bằng chiều cao của 3 củ cà rốt . \n Chiều cao của 1 củ cà rốt là 5 cm . \n C hiều cao của bạn thỏ là: \n 5 + 5 + 5 = 15 (cm) \n Đáp số: 15 cm.,10 cm,12 cm,14 cm,15 cm


In [22]:
# extract đáp án từ explanation
# def extract_answer_letter(explanation):
#     if pd.isna(explanation):
#         return None
#     explanation = str(explanation)
#     m = re.search(r"Đáp án đúng là:\s*([ABCD])", explanation, flags=re.IGNORECASE)
#     if m:
#         return m.group(1).upper()
#     return None

def extract_answer_letter(explanation):
    if pd.isna(explanation):
        return None

    text = str(explanation).strip()

    patterns = [
        # Format phổ biến trong hllj:
        # "Chọn đáp án A."
        # "Chọn đáp án: A"
        r"Chọn\s+đáp\s+án\s*:?\s*([ABCD])\b",

        # Format bạn đang bắt:
        # "Đáp án đúng là: A"
        # "Đáp án đúng là A"
        r"Đáp\s*án\s*đúng\s*là\s*:?\s*([ABCD])\b",

        # Biến thể ngắn hơn:
        # "Đáp án đúng: A"
        r"Đáp\s*án\s*đúng\s*:?\s*([ABCD])\b",

        # Format chung:
        # "Đáp án: A"
        # "Đáp án A"
        r"Đáp\s*án\s*:?\s*([ABCD])\b",

        # Một số mẫu có thể viết:
        # "Ta chọn A"
        r"Ta\s+chọn\s+([ABCD])\b",
    ]

    for p in patterns:
        m = re.search(p, text, flags=re.IGNORECASE)
        if m:
            return m.group(1).upper()

    return None

df_hllj["answer_letter"] = df_hllj["explanation"].apply(extract_answer_letter)

In [23]:
df_hllj.head(3)

,id,grade,url,title,question,choices,explanation,choice_A,choice_B,choice_C,choice_D,answer_letter
0,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,Câu 1: \n \n Độ dài của chiếc bút xoá là:,"[A. 10 cm, B. 8 cm, C. 9 cm, D. 7 cm]",Hướng dẫn giải \n Đáp án đúng là: C \n Quan sát hình vẽ ta thấy độ dài của chiếc bút xoá là 9 cm .,10 cm,8 cm,9 cm,7 cm,C
1,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,"Câu 2: \n \n Nam cao 98 cm, Minh cao 92 cm, Mai cao 88 cm và An cao 89 cm. Bạn thấp nhất là:","[A. Nam, B. Minh, C. Mai, D. An]","Hướng dẫn giải \n Đáp án đúng là: C \n So sánh chiều cao của 4 bạn, ta thấy: 88 < 89 < 92 < 98 . \n Trong 4 số trên số nhỏ nhất là 88, tương ứng với chiều cao của bạn Mai . \n Vậy bạn thấp nhất là bạn Mai .",Nam,Minh,Mai,An,C
2,34a7a20a1cec28e5a0275dec1c9a245e,1,https://khoahoc.vietjack.com/thi-online/bai-tap-on-he-toan-lop-1-chuyen-de-5-do-do-dai-thoi-gian-co-dap-an,Bài tập ôn hè Toán lớp 1 Chuyên đề 5: Đo độ dài – thời gian có đáp án,Câu 3: \n \n Chiều cao của bạn thỏ là:,"[A. 10 cm, B. 12 cm, C. 14 cm, D. 15 cm]",Hướng dẫn giải \n Đáp án đúng là: D \n Chiều cao của bạn thỏ bằng chiều cao của 3 củ cà rốt . \n Chiều cao của 1 củ cà rốt là 5 cm . \n C hiều cao của bạn thỏ là: \n 5 + 5 + 5 = 15 (cm) \n Đáp số: 15 cm.,10 cm,12 cm,14 cm,15 cm,D


In [24]:
df_hllj["subject"] = "Toán"
df_hllj["source_dataset"] = "hllj/vi_grade_school_math_mcq"
df_hllj["sample_id"] = df_hllj["id"].astype(str) + "_" + df_hllj.index.astype(str)

# grade ở đây là cấp/lớp
df_hllj["grade"] = df_hllj["grade"]

df_hllj[["question", "choice_A", "choice_B", "choice_C", "choice_D", "answer_letter", "grade"]].head()

,question,choice_A,choice_B,choice_C,choice_D,answer_letter,grade
0,Câu 1: \n \n Độ dài của chiếc bút xoá là:,10 cm,8 cm,9 cm,7 cm,C,1
1,"Câu 2: \n \n Nam cao 98 cm, Minh cao 92 cm, Mai cao 88 cm và An cao 89 cm. Bạn thấp nhất là:",Nam,Minh,Mai,An,C,1
2,Câu 3: \n \n Chiều cao của bạn thỏ là:,10 cm,12 cm,14 cm,15 cm,D,1
3,Câu 4: \n \n An đi từ nhà đến trường hết 1 giờ. An bắt đầu đi từ nhà đến trường lúc 6 giờ sáng và đến trường lúc:,7 giờ sáng,7 giờ tối,8 giờ sáng,8 giờ tối,A,1
4,Câu 5: \n \n Giờ vào học buổi sáng là 7 giờ. An đến sớm hơn 1 giờ. An đến lớp lúc:,6 giờ,7 giờ,8 giờ,9 giờ,A,1


In [25]:
bad_choice_rows = df_hllj[
    df_hllj[["choice_A", "choice_B", "choice_C", "choice_D"]]
    .map(has_choice_prefix)
    .any(axis=1)
]

bad_choice_rows[["question", "choices", "choice_A", "choice_B", "choice_C", "choice_D", "answer_letter"]].head(20)

,question,choices,choice_A,choice_B,choice_C,choice_D,answer_letter


### Gộp hai nguồn về cùng format trung gian

In [26]:
common_cols = [
    "question",
    "choice_A",
    "choice_B",
    "choice_C",
    "choice_D",
    "answer_letter",
    "subject",
    "grade",
    "source_dataset",
    "sample_id",
]

df_exams_seed = pd.concat(
    [
        df_ro[common_cols].copy(),
        df_hllj[common_cols].copy(),
    ],
    ignore_index=True,
)

df_exams_seed.head()

,question,choice_A,choice_B,choice_C,choice_D,answer_letter,subject,grade,source_dataset,sample_id
0,Đặc điểm nổi bật của nền kinh tế Mĩ sau chiến tranh thế giới thứ 2 là,bị thiệt hại nặng nề về người và của do hậu quả của chiến tranh thế giới thứ hại.,"phát triển mạnh mẽ, vươn lên hàng thứ 2 thế giới sau Liên Xô.",phát triển mạnh mẽ trở thành trung tâm kinh tế - tài chính lớn nhất thế giới.,bị suy giảm nghiêm trọng vì phải lo chi phí cho sản xuất vụ khí.,C,Lịch sử,high school,roshansk23/Vietnam_HighSchool_Exam_Dataset,roshansk_0
1,Nhân tố quan trọng hàng đầu giúp các nước Tây Âu nhanh chóng khôi phục kinh tế sau chiến tranh thế giới thứ hai là gỉ?,Thực hiện các cải cách dân chủ tiến bộ.,Xâm lược trở lại các thuộc địa của mình.,Nhận viện trợ của Mỹ thông qua kế hoạch Mác-san.,Củng cố chính quyền của giai cấp tư sản.,C,Lịch sử,high school,roshansk23/Vietnam_HighSchool_Exam_Dataset,roshansk_1
2,Đâu là nguyên nhân chung cơ bản dẫn đến 3 trung tâm kinh tế tài chính Mĩ – Tây Âu – Nhật Bản khủng hoảng suy thoái kéo dài trong giai đoạn 1973 - 1991?,Tác động của khủng hoảng năng lượng năm 1973.,Sự cạnh tranh quyết liệt của các nước công nghiệp mới.,Sự chi phối ảnh hưởng của trật thế giới 2 cực và chiến tranh lạnh.,Kinh tế Mĩ suy thoái kéo theo kinh tế Nhật Bản và Tây Âu,A,Lịch sử,high school,roshansk23/Vietnam_HighSchool_Exam_Dataset,roshansk_2
3,Việc tìm cách trở lại các thuộc địa cũ sau chiến tranh thế giới thứ 2 của các nước Tây Âu đã ảnh hưởng như thế nào đến Việt Nam?,Thực dân Pháp quay trở lại xâm lược nước ta lần thứ hai buộc nhân dân ta phải đứng lên kháng chiến chống Pháp.,Chính phủ Pháp công nhận Việt Nam là một quốc gia tự do nằm trong khối Liên hiệp Pháp.,"Ngay từ 1945, Pháp – Mỹ đã liên kết lại với nhau để chống cách mạng Việt Nam.","Không ảnh hưởng gì đến Việt Nam vì ngày 2/9/1945, nước Việt Nam Dân chủ Cộng hòa đã tuyên bố thành lập.",A,Lịch sử,high school,roshansk23/Vietnam_HighSchool_Exam_Dataset,roshansk_3
4,Những thắng lợi nào sau đây đánh dấu chủ nghĩa thực dân cũ ở châu Phi cùng hệ thống thuộc địa của nó cơ bản bị tan rã?,"Thắng lợi của nhân dân Môdămbích, Nam Phi.","Thắng lợi của nhân dân Ai Cập, Angiêri.",Thắng lợi của nhân dân Môdămbích và Ănggôla.,"Thắng lợi của nhân dân Ai Cập, Môdămbích.",C,Lịch sử,high school,roshansk23/Vietnam_HighSchool_Exam_Dataset,roshansk_4


In [27]:
df_exams_seed.shape

(20158, 10)

In [28]:
valid_answers = {"A", "B", "C", "D"}

def nonempty(x):
    return pd.notna(x) and str(x).strip() != ""

mask_valid = (
    df_exams_seed["question"].apply(nonempty)
    & df_exams_seed["choice_A"].apply(nonempty)
    & df_exams_seed["choice_B"].apply(nonempty)
    & df_exams_seed["choice_C"].apply(nonempty)
    & df_exams_seed["choice_D"].apply(nonempty)
    & df_exams_seed["answer_letter"].isin(valid_answers)
)

df_exams_seed = df_exams_seed[mask_valid].copy()

# bỏ duplicate rõ ràng theo question + 4 choices + answer
df_exams_seed = df_exams_seed.drop_duplicates(
    subset=["question", "choice_A", "choice_B", "choice_C", "choice_D", "answer_letter"]
).reset_index(drop=True)

df_exams_seed.shape

(11613, 10)

In [29]:
import json

def build_user_content(row):
    return (
        f"Câu hỏi: {str(row['question']).strip()}\n"
        f"A. {str(row['choice_A']).strip()}\n"
        f"B. {str(row['choice_B']).strip()}\n"
        f"C. {str(row['choice_C']).strip()}\n"
        f"D. {str(row['choice_D']).strip()}"
    )

def build_record(row):
    return {
        "messages": [
            {"role": "user", "content": build_user_content(row)},
            {"role": "assistant", "content": f"Đáp án: {row['answer_letter']}"}
        ],
        "metadata": {
            "task": "exams_mcq",
            "source": "public",
            "source_dataset": row["source_dataset"],
            "sample_id": str(row["sample_id"]),
            "difficulty": "medium",
            "split": "train",
            "language": "vi",
            "subject": row["subject"] if pd.notna(row["subject"]) else None,
            "grade": str(row["grade"]) if pd.notna(row["grade"]) else None,
        }
    }

records = [build_record(row) for _, row in df_exams_seed.iterrows()]
len(records)

11613

In [30]:
from pathlib import Path

out_path = Path("seed_exports/exams_mcq_seed.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"[OK] wrote {out_path} | records={len(records)}")

[OK] wrote seed_exports/exams_mcq_seed.jsonl | records=11613


### Recheck with old extract

In [37]:
# recheck
df_exams_seed["source_dataset"].value_counts()

source_dataset
roshansk23/Vietnam_HighSchool_Exam_Dataset    6543
hllj/vi_grade_school_math_mcq                  348
Name: count, dtype: int64

In [42]:
df_exams_seed["subject"].value_counts(dropna=False).head(200)

subject
Lịch sử              1919
Địa lý               1884
Giáo dục công dân    1767
Toán                  348
Toán học              220
Tiếng Anh             205
Hoá học               198
Sinh vật học          175
Vật lý                175
Name: count, dtype: int64

In [43]:
#1) Kiểm tra số câu sau khi explode
print("Raw page rows:", len(df_vi_grade_school_math_mcq))
print("After explode problems:", len(df_hllj_flat))
print("Non-null problems:", df_hllj_flat["question"].notna().sum())

Raw page rows: 2733
After explode problems: 13615
Non-null problems: 13615


In [44]:
# 2) Kiểm tra số mẫu có đủ 4 choices
hllj_has_4_choices = df_hllj["choices"].apply(lambda x: isinstance(x, list) and len(x) == 4)

print("Total hllj flat:", len(df_hllj))
print("Has 4 choices:", hllj_has_4_choices.sum())
print("No 4 choices:", (~hllj_has_4_choices).sum())

Total hllj flat: 13615
Has 4 choices: 8062
No 4 choices: 5553


In [45]:
# 3) Kiểm tra số mẫu parse được answer bằng regex hiện tại
print("Parsed answer_letter:", df_hllj["answer_letter"].notna().sum())
print("Missing answer_letter:", df_hllj["answer_letter"].isna().sum())

Parsed answer_letter: 370
Missing answer_letter: 13245


In [46]:
df_hllj_failed_answer = df_hllj[df_hllj["answer_letter"].isna()].copy()

df_hllj_failed_answer[["question", "choices", "explanation"]].head(20)

,question,choices,explanation
25,Câu 1: \n \n \n Tính: \n a) \n 2 + 3 = … \n 3 + 3 = … \n 1 + 4 = … \n 2 + 4 = … \n b) 1 + 4 + 3 = …,[],Lời giải: \n a. 2 + 3 = 5 \n 3 + 3 = 6 \n 1 + 4 = 5 \n 2 + 4 = 6 \n b. 1 + 4 + 3 = 8 \n 2 + 3 + 3 = 8
26,Câu 2: \n \n \n Điền số thích hợp vào chỗ chấm? \n a) \n … + 4 = 6 \n … = 3 + 2 \n 3 + … = 7 \n 8 = … + 3 \n b) \n \n \n \n \n 1 \n \n \n ….… \n \n \n 3 \n \n \n 4 \n \n \n \n \n 5 < ….. < 7,[],Lời giải
27,"Câu 3: \n \n \n Điền dấu > ,< , = \n 2 + 3 … 4 4 + 0 ….. 5 \n 3 + 1 ….. 2 + 3 3 + 3 ….. 4",[],Lời giải \n \n 2 + 3 > 4 4 + 0 < 5 \n 3 + 1 < 2 + 3 3 + 3 > 4
28,"Câu 4: \n \n \n a. Khoanh vào số bé nhất: 5 , 1 , 4 , 3 , 2 , 7 , 9 \n b. Khoanh vào số lớn nhất: 5 , 7 , 9 , 4 , 2 , 6 , 8",[],
29,Câu 5: \n \n \n Viết các số 8; 4; 2; 5; 6; 9 \n a. Theo thứ tự từ bé đến lớn:……………………………………….. \nb. Theo thứ tự từ lớn đến bé:………………………………………..,[],Lời giải: \n \n Viết các số 8; 4; 2; 5; 6; 9: \n a.Theo thứ tự từ bé đến lớn: 2; 4; 5; 6; 8; 9 \n b.Theo thứ tự từ lớn đến bé: 9; 8; 6; 5; 4; 2
30,"Câu 1: \n \n \n Số bé nhất trong các số 12, 45 87, 52, 97 là:","[A. 12, B. 52, C. 11, D. 97]",Chọn đáp án A.
31,Câu 2: \n \n Số liền trước của số 89 là:,"[A. 87, B. 88, C. 89, D. 90]",Chọn đáp án B.
32,Câu 3: \n \n \n Số tròn chục bé nhất là:,"[A. 80, B. 90, C. 60, D. 10]",Chọn đáp án D.
33,Câu 4: \n \n Số 90 đứng liền sau số nào?,"[A. 89, B. 98, C. 91, D. 92]",Chọn đáp án A.
34,Câu 5: \n \n Kết quả đúng của phép tính: 39cm + 50cm =,"[A. 79cm, B. 89cm, C. 90cm, D. 69 cm]",Chọn đáp án B.


- lý do làm giảm số lượng dòng là do extract đang dùng regex rất hẹp

### Recheck with new extract

In [31]:
hllj_has_4_choices = df_hllj["choices"].apply(lambda x: isinstance(x, list) and len(x) == 4)
hllj_has_answer = df_hllj["answer_letter"].isin({"A", "B", "C", "D"})

print("Total hllj flat:", len(df_hllj))
print("Has 4 choices:", hllj_has_4_choices.sum())
print("Has parsed answer:", hllj_has_answer.sum())
print("Pass both:", (hllj_has_4_choices & hllj_has_answer).sum())

Total hllj flat: 13615
Has 4 choices: 8062
Has parsed answer: 6331
Pass both: 5354


In [32]:
print("Source distribution:")
print(df_exams_seed["source_dataset"].value_counts())

print("\nSubject distribution:")
print(df_exams_seed["subject"].value_counts(dropna=False).head(20))

Source distribution:
source_dataset
roshansk23/Vietnam_HighSchool_Exam_Dataset    6543
hllj/vi_grade_school_math_mcq                 5070
Name: count, dtype: int64

Subject distribution:
subject
Toán                 5070
Lịch sử              1919
Địa lý               1884
Giáo dục công dân    1767
Toán học              220
Tiếng Anh             205
Hoá học               198
Sinh vật học          175
Vật lý                175
Name: count, dtype: int64


In [33]:
df_hllj_final = df_exams_seed[
    df_exams_seed["source_dataset"] == "hllj/vi_grade_school_math_mcq"
]

print("Final hllj records:", len(df_hllj_final))
df_hllj_final[["question", "choice_A", "choice_B", "choice_C", "choice_D", "answer_letter"]].sample(10, random_state=42).head()

Final hllj records: 5070


,question,choice_A,choice_B,choice_C,choice_D,answer_letter
8564,Câu 3: \n \n Dấu cần điền vào ô trông của,>,<,=,Không có dấu nào,B
8383,Câu 2: \n \n Khoanh vào chữ cái trước câu trả lời đúng: 9 tạ 6 kg = …………tạ ; số thích hợp để viết vào chỗ chấm là:,"9,6","9,60","9,06","9,006",C
9859,"Câu 2: \n \n Một người gửi tiết kiệm 3000000 đồng với lãi suất 1,2% mỗi tháng. Tính số tiền người đó nhận được sau 1 tháng.",3030000 đồng,3036000 đồng,3360000 đồng,3630000 đồng,B
8436,"Câu 2: \n \n Phép trừ 712,54 - 48,9 có két quả đúng là:","70,765","223,54","663,64","707,65",C
8531,Câu 2: \n \n Hình hộp chữ nhật có,"6 mặt, 8 đỉnh, 12 cạnh, 1 kích thước","6 mặt, 8 đỉnh, 12 cạnh, 3 kích thước","6 mặt, 8 đỉnh, 12 cạnh, 2 kích thước","6 mặt, 4 đỉnh, 12 cạnh, 2 kích thước",B


In [34]:
#1) Kiểm tra số câu sau khi explode
print("Raw page rows:", len(df_vi_grade_school_math_mcq))
print("After explode problems:", len(df_hllj_flat))
print("Non-null problems:", df_hllj_flat["question"].notna().sum())

Raw page rows: 2733
After explode problems: 13615
Non-null problems: 13615


In [35]:
# 2) Kiểm tra số mẫu có đủ 4 choices
hllj_has_4_choices = df_hllj["choices"].apply(lambda x: isinstance(x, list) and len(x) == 4)

print("Total hllj flat:", len(df_hllj))
print("Has 4 choices:", hllj_has_4_choices.sum())
print("No 4 choices:", (~hllj_has_4_choices).sum())

Total hllj flat: 13615
Has 4 choices: 8062
No 4 choices: 5553


In [36]:
# 3) Kiểm tra số mẫu parse được answer bằng regex hiện tại
print("Parsed answer_letter:", df_hllj["answer_letter"].notna().sum())
print("Missing answer_letter:", df_hllj["answer_letter"].isna().sum())

Parsed answer_letter: 6331
Missing answer_letter: 7284


In [37]:
df_hllj_failed_answer = df_hllj[df_hllj["answer_letter"].isna()].copy()

df_hllj_failed_answer[["question", "choices", "explanation"]].head(20)

,question,choices,explanation
25,Câu 1: \n \n \n Tính: \n a) \n 2 + 3 = … \n 3 + 3 = … \n 1 + 4 = … \n 2 + 4 = … \n b) 1 + 4 + 3 = …,[],Lời giải: \n a. 2 + 3 = 5 \n 3 + 3 = 6 \n 1 + 4 = 5 \n 2 + 4 = 6 \n b. 1 + 4 + 3 = 8 \n 2 + 3 + 3 = 8
26,Câu 2: \n \n \n Điền số thích hợp vào chỗ chấm? \n a) \n … + 4 = 6 \n … = 3 + 2 \n 3 + … = 7 \n 8 = … + 3 \n b) \n \n \n \n \n 1 \n \n \n ….… \n \n \n 3 \n \n \n 4 \n \n \n \n \n 5 < ….. < 7,[],Lời giải
27,"Câu 3: \n \n \n Điền dấu > ,< , = \n 2 + 3 … 4 4 + 0 ….. 5 \n 3 + 1 ….. 2 + 3 3 + 3 ….. 4",[],Lời giải \n \n 2 + 3 > 4 4 + 0 < 5 \n 3 + 1 < 2 + 3 3 + 3 > 4
28,"Câu 4: \n \n \n a. Khoanh vào số bé nhất: 5 , 1 , 4 , 3 , 2 , 7 , 9 \n b. Khoanh vào số lớn nhất: 5 , 7 , 9 , 4 , 2 , 6 , 8",[],
29,Câu 5: \n \n \n Viết các số 8; 4; 2; 5; 6; 9 \n a. Theo thứ tự từ bé đến lớn:……………………………………….. \nb. Theo thứ tự từ lớn đến bé:………………………………………..,[],Lời giải: \n \n Viết các số 8; 4; 2; 5; 6; 9: \n a.Theo thứ tự từ bé đến lớn: 2; 4; 5; 6; 8; 9 \n b.Theo thứ tự từ lớn đến bé: 9; 8; 6; 5; 4; 2
37,Câu 3: \n \n \n Viết số còn thiếu vào chỗ chấm: \n a) Số … là số liền trước của số 49. \n b) Số …là số liền sau của số 58.,[],a) Số 48 là số liền trước của số 49. \n b) Số 59 là số liền sau của số 58.
38,"Câu 4: \n \n \n Đúng ghi Đ, sai ghi S vào ô trống: \n \n 12 + 13 = 25 ☐ 33 – 11 = 21 ☐ 45 + 10 = 55 ☐ 89 – 47 = 42 ☐",[],12 + 13 = 25 [Đ] 33 – 11 = 21 [S] 45 + 10 = 55 [Đ] 89 – 47 = 42 [Đ]
39,Câu 5: \n \n \n Đọc tờ lịch dưới đây và điền số thích hợp vào chỗ chấm: \n \n \n Hôm nay là thứ …. ngày … tháng … năm 2021.,[],Hôm nay là thứ ba ngày 22 tháng 6 năm 2021.
40,Câu 1: \n \n \n Điền số thích hợp vào ô trống:,[],Lời giải:
41,Câu 2: \n \n \n Tính: \n 4 + 2 = … 6 – 5 = … 2 + 7 = … 10 – 0 = … \n 6 + 1 = … 9 – 5 = … 2 + 5 = … 4 + 4 = …,[],Lời giải: \n \n 4 + 2 = 6 6 – 5 = 1 2 + 7 = 9 10 – 0 = 10 \n 6 + 1 = 7 9 – 5 = 4 2 + 5 = 7 4 + 4 = 8


In [38]:
df_exams_seed.shape

(11613, 10)

In [39]:
bad_sample = df_exams_seed[
    df_exams_seed["question"].astype(str).str.contains(
        "Trong các số sau số nào chia hết cho 2",
        regex=False,
        na=False
    )
]

bad_sample[["question", "choice_A", "choice_B", "choice_C", "choice_D", "answer_letter"]]

,question,choice_A,choice_B,choice_C,choice_D,answer_letter
7338,Câu 2: \n \n Trong các số sau số nào chia hết cho 2 nhưng không chia hết cho 5,48405,46254,90450,17309,B
7385,Câu 5: \n \n Trong các số sau số nào chia hết cho 2: 3457; 4568; 2229; 2355,3457,4568,2229,2355,B
7447,Câu 3: \n \n Trong các số sau số nào chia hết cho 2 là:,605,1207,3642,2401,C
7509,Câu 3: \n \n Trong các số sau số nào chia hết cho 2 và 5 ?,456 570,456 705,456 057,456 507,A
7560,Câu 3: \n \n Trong các số sau số nào chia hết cho 2 là,605,1207,3642,2401,C
9732,Câu 3: \n \n Trong các số sau số nào chia hết cho 2?,1235,1331,2469,1998,D


In [40]:
prefix_pattern = r"^\s*[A-Da-d]\s*[\.\)\:\-]\s*"

bad_prefix_rows = df_exams_seed[
    df_exams_seed[["choice_A", "choice_B", "choice_C", "choice_D"]]
    .apply(lambda col: col.astype(str).str.contains(prefix_pattern, regex=True, na=False))
    .any(axis=1)
]

print("Rows still having choice prefix:", len(bad_prefix_rows))

bad_prefix_rows[
    ["question", "choice_A", "choice_B", "choice_C", "choice_D", "answer_letter", "source_dataset"]
].head(20)

Rows still having choice prefix: 0


,question,choice_A,choice_B,choice_C,choice_D,answer_letter,source_dataset


In [41]:
records = [build_record(row) for _, row in df_exams_seed.iterrows()]

out_path = Path("seed_exports/exams_mcq_seed.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"[OK] wrote {out_path} | records={len(records)}")

[OK] wrote seed_exports/exams_mcq_seed.jsonl | records=11613
